# Phase 6 · Data analysis — Abstractive Legal Summarization

Computes what the Phase 6 design depends on and writes two small files to send back:
- `P6_analysis/p6_stats.json` — sizes, document and summary lengths (characters / words / BPE tokens), compression ratios, how much of each document fits in 512–2048 tokens, where in the document the summary's content comes from, how abstractive the summaries are, ROUGE of simple extractive baselines (the floor to beat), split overlaps, and the context lengths of the four Phase-1 models.
- `P6_analysis/p6_samples.json` — a few documents (beginning and end) with their reference summaries.

**Kaggle setup:** **CPU** is enough · Add-ons → Secrets: **`HF`** · Internet on · Add Input → **`rohitbatra2004/cs781-phase1-tokenized-corpus`** (tokenizer) and the four Phase-1 model datasets **`rohitbatra2004/phase111`** (A), **`rohitbatra2004/arch-data-1`** (B), **`rohitbatra2004/phase133`** (C), **`ruchikhemka/data-archd`** (D) — only their small config files are read. Expected time ≈ 15–25 minutes. Download both files from `P6_analysis/` and send them.

In [ ]:
# Cell 1 — Imports, output folder and random seed.
import json, glob, random, hashlib, re
from pathlib import Path
from collections import Counter
import numpy as np
from tokenizers import Tokenizer
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient

DATA_DIR = Path("/kaggle/working/data")
OUT_DIR = Path("/kaggle/working/P6_analysis")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(42)
SAMPLE = 1000                       # documents used for the slower content / baseline analyses
print("Outputs:", OUT_DIR)

In [ ]:
# Cell 2 — Download Phase_6_Summ from Hugging Face using the Kaggle secret "HF".
snapshot_download(repo_id="Exploration-Lab/CS-781-Capstone", repo_type="dataset", token=UserSecretsClient().get_secret("HF"),
                  local_dir=str(DATA_DIR), allow_patterns=["Phase_6_Summ/**"])
P6_DIR = DATA_DIR / "Phase_6_Summ"
for f in sorted(P6_DIR.rglob("*")):
    if f.is_file():
        print(f"{f.relative_to(P6_DIR)}: {f.stat().st_size / 1e6:.1f} MB")

In [ ]:
# Cell 3 — Load train and test, show the record structure, and count empty or duplicated records.
def load(path):
    data = json.load(open(path))
    return data if isinstance(data, list) else [dict(v, id=k) for k, v in data.items()]

train, test = load(P6_DIR / "Train/train.json"), load(P6_DIR / "Test/test.json")
stats = {"n_cases": {"train": len(train), "test": len(test)},
         "fields": {"train": {k: type(v).__name__ for k, v in train[0].items()},
                    "test": {k: type(v).__name__ for k, v in test[0].items()}}}
for name, docs in (("train", train), ("test", test)):
    print(f"{name}: {len(docs)} records | fields {stats['fields'][name]}")
    print("   example:", {k: (v[:100] + "…" if isinstance(v, str) and len(v) > 100 else v) for k, v in docs[0].items()})
key = lambda t: hashlib.md5(" ".join(t.split()).encode()).hexdigest()
train_hashes = Counter(key(d["document"]) for d in train)
stats["quality"] = {
    "empty_documents": {"train": sum(not d["document"].strip() for d in train), "test": sum(not d["document"].strip() for d in test)},
    "empty_summaries_train": sum(not d["summary"].strip() for d in train),
    "duplicate_documents_within_train": sum(c - 1 for c in train_hashes.values() if c > 1),
    "test_documents_also_in_train": sum(key(d["document"]) in train_hashes for d in test),
    "duplicate_ids": {"train": len(train) - len({d["id"] for d in train}), "test": len(test) - len({d["id"] for d in test})},
    "share_lowercase_documents": {n: round(float(np.mean([d["document"] == d["document"].lower() for d in docs[:2000]])), 3)
                                  for n, docs in (("train", train), ("test", test))},
}
print(json.dumps(stats["quality"], indent=1))

In [ ]:
# Cell 4 — Lengths in characters and words: documents (train, test), summaries (train), and the compression ratio.
def pct(xs, ps=(0, 5, 10, 25, 50, 75, 90, 95, 99, 100)):
    arr = np.array(xs, dtype=float)
    return {f"p{p}": int(np.percentile(arr, p)) for p in ps} | {"mean": int(arr.mean())}

words = lambda t: len(t.split())
stats["words"] = {"document_train": pct([words(d["document"]) for d in train]), "document_test": pct([words(d["document"]) for d in test]),
                  "summary_train": pct([words(d["summary"]) for d in train])}
stats["chars"] = {"document_train": pct([len(d["document"]) for d in train]), "summary_train": pct([len(d["summary"]) for d in train])}
ratio = [words(d["summary"]) / max(words(d["document"]), 1) for d in train]
stats["summary_to_document_word_ratio"] = {f"p{p}": round(float(np.percentile(ratio, p)), 3) for p in (10, 25, 50, 75, 90)}
for k, v in stats["words"].items():
    print(f"{k:16s} words {v}")
print("summary / document length ratio:", stats["summary_to_document_word_ratio"])

In [ ]:
# Cell 5 — BPE-token lengths (Phase-1 tokenizer) and how many documents fit completely in 512 / 1,024 / 1,536 / 2,048 tokens.
tokenizer = Tokenizer.from_file(sorted(glob.glob("/kaggle/input/**/tokenizer/tokenizer.json", recursive=True))[0])
def token_lengths(texts):
    out = []
    for s in range(0, len(texts), 256):
        out += [len(e.ids) for e in tokenizer.encode_batch(texts[s:s + 256])]
    return np.array(out)

doc_tokens = {"train": token_lengths([d["document"] for d in train]), "test": token_lengths([d["document"] for d in test])}
sum_tokens = token_lengths([d["summary"] for d in train])
stats["tokens"] = {"document_train": pct(doc_tokens["train"]), "document_test": pct(doc_tokens["test"]), "summary_train": pct(sum_tokens)}
stats["documents_fully_within"] = {name: {str(k): round(float((v <= k).mean()), 3) for k in (512, 1024, 1536, 2048, 4096)}
                                   for name, v in doc_tokens.items()}
stats["summaries_fully_within"] = {str(k): round(float((sum_tokens <= k).mean()), 3) for k in (128, 256, 384, 512, 768, 1024)}
for k, v in stats["tokens"].items():
    print(f"{k:16s} tokens {v}")
print("documents fully within k tokens:", stats["documents_fully_within"])
print("summaries fully within k tokens:", stats["summaries_fully_within"])

In [ ]:
# Cell 6 — Where does the summary's content come from? For each summary sentence, the document sentence with the highest
# word overlap (recall) is found; its relative position (tenths of the document) is counted. Also: how abstractive
# the summaries are (share of summary words / bigrams that never appear in the document).
SENT_SPLIT = re.compile(r"(?<=[.?!])\s+(?=[A-Z0-9(\"'])")
TOKEN = re.compile(r"[a-z0-9]+")
def sentences(text):
    return [s for s in SENT_SPLIT.split(text.strip()) if len(s.split()) >= 3]
def toks(text):
    return TOKEN.findall(text.lower())

sample = random.sample(train, min(SAMPLE, len(train)))
position_hist = np.zeros(10)
novel_unigrams, novel_bigrams, n_summary_sentences, best_overlap = [], [], [], []
for d in sample:
    doc_sents = sentences(d["document"])
    if not doc_sents:
        continue
    doc_sets = [set(toks(s)) for s in doc_sents]
    doc_words = toks(d["document"]); doc_vocab = set(doc_words); doc_bigrams = set(zip(doc_words, doc_words[1:]))
    sw = toks(d["summary"])
    if sw:
        novel_unigrams.append(np.mean([w not in doc_vocab for w in sw]))
        bigrams = list(zip(sw, sw[1:]))
        if bigrams:
            novel_bigrams.append(np.mean([b not in doc_bigrams for b in bigrams]))
    summ_sents = sentences(d["summary"])
    n_summary_sentences.append(len(summ_sents))
    for s in summ_sents:
        ss = set(toks(s))
        if not ss:
            continue
        scores = [len(ss & ds) / len(ss) for ds in doc_sets]
        j = int(np.argmax(scores))
        best_overlap.append(scores[j])
        position_hist[min(9, int(10 * j / len(doc_sents)))] += 1
stats["summary_content_position_deciles"] = (position_hist / max(position_hist.sum(), 1)).round(3).tolist()
stats["abstractiveness"] = {"novel_unigram_share_mean": round(float(np.mean(novel_unigrams)), 3),
                            "novel_bigram_share_mean": round(float(np.mean(novel_bigrams)), 3),
                            "best_matching_document_sentence_recall_median": round(float(np.median(best_overlap)), 3),
                            "summary_sentences_per_case": pct(n_summary_sentences, (10, 25, 50, 75, 90))}
print("Share of summary sentences whose best source lies in each tenth of the document:", stats["summary_content_position_deciles"])
print("Abstractiveness:", stats["abstractiveness"])

In [ ]:
# Cell 7 — ROUGE-1 / ROUGE-L F1 of simple extractive baselines on the same sample (the floor a trained model should beat):
# lead (first sentences), tail (last sentences), Luhn-style "important" sentences, each cut to the median summary length;
# plus a greedy oracle (best achievable by copying whole sentences).
def rouge1_f(ref, hyp):
    r, h = Counter(ref), Counter(hyp)
    overlap = sum((r & h).values())
    return 0.0 if overlap == 0 else 2 * overlap / (len(ref) + len(hyp))
def lcs(a, b):
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b, 1):
            cur.append(prev[j - 1] + 1 if x == y else max(prev[j], cur[j - 1]))
        prev = cur
    return prev[-1]
def rougeL_f(ref, hyp):
    ref, hyp = ref[:600], hyp[:600]                      # cap the O(n·m) LCS on very long texts
    l = lcs(ref, hyp)
    return 0.0 if l == 0 else 2 * l / (len(ref) + len(hyp))
def take_words(sents, n_words):
    out, total = [], 0
    for s in sents:
        if total >= n_words:
            break
        out.append(s); total += len(s.split())
    return " ".join(out)

TARGET_WORDS = stats["words"]["summary_train"]["p50"]
baseline_scores = {k: {"rouge1": [], "rougeL": []} for k in ("lead", "tail", "luhn_important", "oracle_greedy")}
for d in sample[:500]:
    doc_sents = sentences(d["document"])
    if not doc_sents:
        continue
    ref = toks(d["summary"])
    freq = Counter(w for w in toks(d["document"]) if len(w) > 3)
    score = [sum(freq[w] for w in set(toks(s))) / (len(toks(s)) + 1) for s in doc_sents]
    important = [doc_sents[i] for i in sorted(np.argsort(score)[::-1][:40])]
    chosen, oracle_best = [], 0.0                          # greedy oracle on ROUGE-1
    for _ in range(15):
        gains = [(rouge1_f(ref, toks(" ".join(chosen + [s]))), s) for s in doc_sents[:200] if s not in chosen]
        if not gains:
            break
        g, s = max(gains)
        if g <= oracle_best:
            break
        chosen.append(s); oracle_best = g
    hyps = {"lead": take_words(doc_sents, TARGET_WORDS), "tail": take_words(doc_sents[::-1], TARGET_WORDS),
            "luhn_important": take_words(important, TARGET_WORDS), "oracle_greedy": " ".join(chosen)}
    for k, h in hyps.items():
        baseline_scores[k]["rouge1"].append(rouge1_f(ref, toks(h))); baseline_scores[k]["rougeL"].append(rougeL_f(ref, toks(h)))
stats["extractive_baselines_rouge"] = {k: {m: round(float(np.mean(v)), 4) for m, v in s.items()} for k, s in baseline_scores.items()}
stats["extractive_baselines_target_words"] = TARGET_WORDS
for k, v in stats["extractive_baselines_rouge"].items():
    print(f"  {k:15s}: ROUGE-1 F1 {v['rouge1']:.4f} | ROUGE-L F1 {v['rougeL']:.4f}")

In [ ]:
# Cell 8 — How summaries are written: most common opening words and summary-level patterns (template-like openings?).
openings = Counter(" ".join(d["summary"].split()[:4]) for d in train)
first_words = Counter(d["summary"].split()[0] if d["summary"].split() else "" for d in train)
stats["summary_openings_top"] = openings.most_common(20)
stats["summary_first_word_top"] = first_words.most_common(15)
print("Most common summary openings:", openings.most_common(10))
print("Most common first words:", first_words.most_common(10))

In [ ]:
# Cell 9 — Context limits of the four Phase-1 models (read from their config / metadata files only).
model_limits = {}
for arch, pattern in (("A", "architecture_A/model_A_metadata.json"), ("B", "architecture_B/model_B_metadata.json"),
                      ("C", "architecture_C/model_C_metadata.json")):
    found = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if found:
        cfg = json.load(open(found[0]))["model_config"]
        model_limits[arch] = {k: v for k, v in cfg.items() if "context" in k or k in ("d_model", "n_layers", "n_encoder_layers",
                                                                                      "n_decoder_layers", "hidden_size", "num_layers")}
d_found = sorted(glob.glob("/kaggle/input/**/architecture_D/checkpoints/best.pt", recursive=True))
if d_found:
    import torch
    cfg = torch.load(d_found[0], map_location="cpu", mmap=True, weights_only=False)["model_config"]
    model_limits["D"] = {"context_length": cfg.get("context_length"), "channels": cfg["channels"], "dilations": cfg["dilations"],
                         "receptive_field": 1 + 2 * (cfg["kernel_size"] - 1) * sum(cfg["dilations"])}
stats["phase1_model_limits"] = model_limits
print(json.dumps(model_limits, indent=1))
missing = {"A", "B", "C", "D"} - set(model_limits)
if missing:
    print("Not attached (optional, only for this cell):", sorted(missing))

In [ ]:
# Cell 10 — Save the statistics and a few example documents with their reference summaries.
samples = {"train": [{"id": str(d["id"]), "doc_words": words(d["document"]), "summary_words": words(d["summary"]),
                      "document_start": d["document"][:1500], "document_end": d["document"][-800:], "summary": d["summary"][:3000]}
                     for d in random.sample(train, 4)],
           "test": [{"id": str(d["id"]), "doc_words": words(d["document"]), "document_start": d["document"][:800]}
                    for d in random.sample(test, 2)]}
json.dump(stats, open(OUT_DIR / "p6_stats.json", "w"), indent=2)
json.dump(samples, open(OUT_DIR / "p6_samples.json", "w"), indent=2)
for f in sorted(OUT_DIR.iterdir()):
    print(f"{f.name}: {f.stat().st_size / 1e3:.1f} KB")
print("Download both files from the Output panel (P6_analysis/) and send them.")